# Decorators aur Namespaces — Complete Guide 
---

## Table of Contents

1. [Part 1: Namespaces](#part-1-namespaces)
   - [1.1 Namespace kya hota hai](#11-namespace-kya-hota-hai)
   - [1.2 Namespace ke Types](#12-namespace-ke-types)
   - [1.3 LEGB Rule — Python Variable Kaise Dhoondta Hai](#13-legb-rule--python-variable-kaise-dhoondta-hai)
   - [1.4 `global` Keyword](#14-global-keyword)
   - [1.5 `nonlocal` Keyword](#15-nonlocal-keyword)
   - [1.6 `globals()` aur `locals()`](#16-globals-aur-locals)
   - [1.7 Namespace ki Lifecycle](#17-namespace-ki-lifecycle)
   - [1.8 Scope vs Namespace — Difference](#18-scope-vs-namespace--difference)
2. [Part 2: Decorators](#part-2-decorators)
   - [2.1 Prerequisite: Functions First-Class Citizens Hain](#21-prerequisite-functions-first-class-citizens-hain)
   - [2.2 Prerequisite: Closures](#22-prerequisite-closures)
   - [2.3 Decorator kya hota hai](#23-decorator-kya-hota-hai)
   - [2.4 Apna Pehla Decorator Banana](#24-apna-pehla-decorator-banana)
   - [2.5 `@` Syntax Actually Karta Kya Hai](#25--syntax-actually-karta-kya-hai)
   - [2.6 Arguments Wale Functions Decorate Karna](#26-arguments-wale-functions-decorate-karna)
   - [2.7 `functools.wraps` — Metadata Save Karna](#27-functoolswraps--metadata-save-karna)
   - [2.8 Decorators with Parameters (Decorator Factory)](#28-decorators-with-parameters-decorator-factory)
   - [2.9 Multiple Decorators Chain Karna](#29-multiple-decorators-chain-karna)
   - [2.10 Class-based Decorators](#210-class-based-decorators)
   - [2.11 Built-in Decorators](#211-built-in-decorators)
   - [2.12 Real-World Use Cases](#212-real-world-use-cases)
   - [2.13 Common Pitfalls](#213-common-pitfalls)
   - [Quick Recap](#quick-recap)

---

# Part 1: Namespaces

## 1.1 Namespace kya hota hai

**Namespace** ek **mapping (dictionary jaisi)** hoti hai jo **naam (variable/function/class name) ko uske actual object se jodti hai**. Jab tum `x = 5` likhte ho, Python internally kahin na kahin ek entry banata hai: `"x" → 5`. Ye entry jis "container" mein store hoti hai, wahi namespace hai.

A namespace is a space that holds names(identifiers).Programmatically speaking, namespaces are dictionary of identifiers(keys) and their objects(values)

There are 4 types of namespaces:
- Builtin Namespace
- Global Namespace
- Enclosing Namespace
- Local Namespace

**Real life analogy:** Socho ek **school** hai jisme har class (Class 9-A, Class 9-B) ke apne roll numbers hain. Class 9-A ka Roll No. 5 ek alag student hai, Class 9-B ka Roll No. 5 bilkul alag student hai — same "naam" (Roll No. 5) do alag **namespaces** (Class 9-A, Class 9-B) mein do alag cheezon ko refer karta hai. Bilkul waise hi Python mein ek hi naam (`x`) alag-alag namespaces mein alag-alag values ko point kar sakta hai.

```python
x = 10   # global namespace mein x → 10(global scope)

def my_function():
    x = 20    # local namespace mein x → 20 (bilkul alag entry, global wale se koi lena dena nahi) local scope
    print(x)  # 20

my_function()
print(x)   # 10 (global wala x change nahi hua)
```
```mermaid
flowchart TD
    subgraph Global["Global Namespace"]
        G1["x ➔ 10"]
    end
    
    subgraph Local["Local Namespace"]
        L1["x ➔ 20<br>(my_function ke andar)"]
    end

    %% Visual Styling for Dark Mode
    style Global fill:#1e1e1e,stroke:#007acc,stroke-width:2px,color:#fff
    style Local fill:#1e1e1e,stroke:#4ec9b0,stroke-width:2px,color:#fff
    style G1 fill:#2d2d2d,stroke:#3c3c3c,color:#dcdcdc
    style L1 fill:#2d2d2d,stroke:#3c3c3c,color:#dcdcdc
```


---

## 1.2 Namespace ke Types

Python mein **4 tarah ke namespaces** hote hain, jo ek hierarchy mein arrange hote hain:

| Namespace | Kya hai | Kab banta hai | Kab destroy hota hai |
|---|---|---|---|
| **Built-in** | Python ke apne built-in functions/names (`print`, `len`, `str`) | Python interpreter start hote hi | Interpreter band hone pe |
| **Global (Module)** | Har `.py` file/module ka apna top-level namespace | Module import/run hote hi | Program/module khatam hone pe |
| **Enclosing** | Kisi **outer function** ka namespace, jab ek function doosre function ke andar ho (nested functions) | Outer function call hote hi | Outer function khatam hone pe |
| **Local** | Kisi function ke **andar** ka namespace | Function call hote hi | Function return hote hi |

```mermaid
flowchart TD

    A["🔧 Built-in Namespace<br/>(print, len, str, int...)"]
    --> B["🌍 Global / Module Namespace<br/>(file-level variables)"]

    B --> C["📦 Enclosing Namespace<br/>(outer function ka scope)"]

    C --> D["🎯 Local Namespace<br/>(current function ka scope)"]

    %% Styling
    classDef builtin fill:#f0e9ff,stroke:#7c3aed,color:#111827,stroke-width:2px
    classDef global fill:#eaf2ff,stroke:#3b82f6,color:#111827,stroke-width:2px
    classDef enclosing fill:#fff8e6,stroke:#f59e0b,color:#111827,stroke-width:2px
    classDef local fill:#eaf8ef,stroke:#22c55e,color:#111827,stroke-width:2px

    class A builtin
    class B global
    class C enclosing
    class D local

    %% Link styling
    linkStyle default stroke:#64748b,stroke-width:2px
```

---

## 1.3 LEGB Rule — Python Variable Kaise Dhoondta Hai

Jab tum kisi variable ko **use** karte ho, Python usko dhoondhne ke liye ek **fixed order** follow karta hai — isko **LEGB Rule** kehte hain:

**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

Jaise hi variable mil jaata hai, Python wahin ruk jaata hai — aage nahi dhoondhta.

```python
x = "global x"                    # Global

def outer():
    x = "enclosing x"             # Enclosing (outer function ka apna scope)

    def inner():
        x = "local x"             # Local
        print(x)                  # "local x" — Local mein mil gaya, yahin ruk gaya

    inner()

outer()
```

Agar `inner()` ke andar `x` na hoti, to Python **Enclosing** mein dhoondhta (`outer` ka `x`). Agar wahan bhi na milta, to **Global** mein, aur wahan bhi na milta to **Built-in** mein, aur agar wahan bhi nahi mila to `NameError`.
```mermaid
flowchart TD

    Start(["inner() ke andar 'x' use hua"])
    
    Start --> L{"Local namespace<br/>mein 'x' hai?"}
    L -->|Haan| F1["✅ Yahi use karo<br/>ruk jao"]
    L -->|Nahi| E{"Enclosing namespace<br/>mein 'x' hai?"}
    
    E -->|Haan| F2["✅ Yahi use karo<br/>ruk jao"]
    E -->|Nahi| G{"Global namespace<br/>mein 'x' hai?"}
    
    G -->|Haan| F3["✅ Yahi use karo<br/>ruk jao"]
    G -->|Nahi| B{"Built-in namespace<br/>mein 'x' hai?"}
    
    B -->|Haan| F4["✅ Yahi use karo"]
    B -->|Nahi| Error["❌ NameError:<br/>name 'x' is not defined"]

    %% Styling
    classDef start fill:#e8e3ff,stroke:#7c3aed,color:#111827,stroke-width:2px
    classDef decision fill:#eaf2ff,stroke:#3b82f6,color:#111827,stroke-width:2px
    classDef success fill:#eaf8ef,stroke:#22c55e,color:#111827,stroke-width:2px
    classDef error fill:#fff0f0,stroke:#ef4444,color:#111827,stroke-width:2px

    class Start start
    class L,E,G,B decision
    class F1,F2,F3,F4 success
    class Error error

    %% Link styling
    linkStyle default stroke:#64748b,stroke-width:2px
```
---

## 1.4 `global` Keyword

Default behavior: function ke andar agar tum kisi variable ko **assign** karte ho, Python use **automatically local** maan leta hai — chahe bahar (global mein) same naam ka variable exist bhi karta ho.

```python
counter = 0

def increment():
    counter += 1   # ❌ UnboundLocalError!
    # Python ne dekha counter = ... ho raha hai, to isko LOCAL maan liya
    # Par local counter abhi tak define hi nahi hua, isliye error

increment()
```

Isko fix karne ke liye `global` keyword use karo — ye Python ko batata hai "is function ke andar jab bhi is naam ko assign karo, global wale ko hi modify karna, naya local mat banana":

```python
counter = 0

def increment():
    global counter    # ab Python jaanta hai ki 'counter' global wala hi hai
    counter += 1

increment()
print(counter)   # 1
```

---

## 1.5 `nonlocal` Keyword

`global` sirf **global namespace** tak jaata hai. Par agar tumhe **enclosing (outer function) ka variable modify** karna hai (global nahi), to `nonlocal` use hota hai:

```python
def outer():
    count = 0

    def inner():
        nonlocal count   # outer() ke 'count' ko refer karo, naya local mat banao
        count += 1
        print(count)

    inner()
    inner()
    print("Final:", count)

outer()
# Output: 1, 2, Final: 2
```

```mermaid
flowchart LR
    subgraph "global keyword"
    A["Function ke andar se\nGlobal namespace modify karna"]
    end
    subgraph "nonlocal keyword"
    B["Function ke andar se\nEnclosing (outer function)\nka namespace modify karna"]
    end
```

---

## 1.6 `globals()` aur `locals()`

Ye do built-in functions actual namespace ko **dictionary ki tarah dikha dete hain**:

```python
x = 10

def my_func():
    y = 20
    print(locals())    # {'y': 20} — sirf function ke andar ke variables

print(globals())        # poori dictionary — module-level saare naam (x sameet)
```

> **Debugging tip:** Agar kabhi confuse ho ki koi variable kis scope mein hai, `locals()` ya `globals()` print karke dekh sakte ho.

---

## 1.7 Namespace ki Lifecycle

Namespace **hamesha zinda nahi rehte** — ye create hote hain aur destroy bhi hote hain:

```mermaid
sequenceDiagram
    participant P as Program Start
    participant G as Global Namespace
    participant F as Function Call
    participant L as Local Namespace

    P->>G: Module load hote hi bana
    F->>L: Function call hote hi naya Local Namespace bana
    Note over L: Function ke andar ka code chalta hai
    F->>L: Function return hote hi Local Namespace DESTROY
    Note over G: Global Namespace tab tak zinda rehta hai\njab tak program/module chal raha hai
```

> Yehi wajah hai ki function ke andar declare kiya hua variable function khatam hote hi "gayab" ho jaata hai — uska local namespace hi khatam ho gaya.

---

## 1.8 Scope vs Namespace — Difference

Ye dono terms confuse karte hain, chalo clear karte hain:

| | Namespace | Scope |
|---|---|---|
| **Kya hai** | Naam → object ki actual **mapping** (dictionary jaisi cheez) | Code ka wo **region/area** jaha se ek particular namespace **accessible** hai |
| **Analogy** | Phone ki contact list (naam → number) | Wo area jaha tak tumhara signal/network pahunchta hai |

> Simple tareeke se: **Namespace** batata hai "kya store hai", **Scope** batata hai "kahan se access kar sakte ho".

---

# Part 2: Decorators

## 2.1 Prerequisite: Functions First-Class Citizens Hain

Decorators samajhne se pehle ek cheez clearly samajh lo: Python mein **functions bhi ek normal object hi hain** (jaise int, string, list) — inhe variable mein store kar sakte ho, doosre function ko as argument bhej sakte ho, aur function se **return** bhi kar sakte ho.

```python
def greet():
    return "Hello!"

# Function ko variable mein store karna
say_hello = greet
print(say_hello())    # "Hello!" — function object copy ho gaya

# Function ko argument ki tarah pass karna
def call_function(func):
    return func()

print(call_function(greet))   # "Hello!"

# Function ko return karna
def outer_function():
    def inner_function():
        return "Inner se aaya!"
    return inner_function    # NOTE: () nahi lagaya — function object return kiya, call nahi kiya

my_func = outer_function()
print(my_func())    # "Inner se aaya!"
```

Isi property ko **"functions are first-class objects"** kehte hain — ye poore decorators concept ki **foundation** hai.

---

## 2.2 Prerequisite: Closures

**Closure** hota hai jab ek **inner function** apne **outer function ke variables ko "yaad" rakhta hai**, chahe outer function apna kaam khatam karke return ho chuka ho.

```python
def outer_function(msg):
    def inner_function():
        print(msg)    # 'msg' outer_function ka parameter hai, par inner isko "yaad" rakhta hai
    return inner_function

hello_func = outer_function("Namaste!")
hello_func()    # "Namaste!" — outer_function to kabka khatam ho chuka, phir bhi msg yaad hai!
```

```mermaid
flowchart TD
    A["outer_function('Namaste!') call hua"] --> B["inner_function define hua,\nmsg='Namaste!' ko capture kar liya"]
    B --> C["outer_function return ho gaya\naur khatam ho gaya"]
    C --> D["Par inner_function ke paas\nabhi bhi msg ki memory hai\n(ye hai CLOSURE)"]
    D --> E["hello_func() call karne pe\nmsg='Namaste!' print hota hai"]
```

> **Decorators closures pe hi based hote hain** — decorator ek function hai jo doosre function ko "yaad" rakhta hai aur uske around extra behavior add karta hai.

---

## 2.3 Decorator kya hota hai

**Decorator** ek function hai jo **doosre function ko input leta hai, usme kuch extra functionality "wrap" karta hai, aur ek naya (modified) function return karta hai** — **bina original function ka code change kiye**.

**Real life analogy:** Socho tumhare paas ek **plain gift box** hai. Decorator ek **gift wrap** hai — box ke andar ka saaman (original function) same rehta hai, bas upar se ek extra sundar packaging (extra behavior) add ho jaati hai.

```mermaid
flowchart LR
    A["Original Function\n(plain gift)"] -->|Decorator apply hota hai| B["Wrapped Function\n(gift + wrapping paper)"]
    B --> C["Jab call hoga:\nExtra behavior + Original function\ndono chalenge"]
```

---

## 2.4 Apna Pehla Decorator Banana

Chalo step-by-step ek simple decorator banate hain jo function call hone se pehle aur baad mein ek message print kare:

```python
def my_decorator(func):          # Step 1: decorator ek function leta hai
    def wrapper():                # Step 2: ek "wrapper" function define karo (closure!)
        print("Function chalne se PEHLE...")
        func()                     # Step 3: original function ko andar se call karo
        print("Function chalne ke BAAD...")
    return wrapper                 # Step 4: wrapper function return karo (call nahi kiya, sirf return)

def say_hello():
    print("Hello Samarth!")

# Manually decorator apply karna:
decorated_function = my_decorator(say_hello)
decorated_function()
```

**Output:**
```
Function chalne se PEHLE...
Hello Samarth!
Function chalne ke BAAD...
```

---

## 2.5 `@` Syntax Actually Karta Kya Hai

Upar wala manual tareeka (`decorated_function = my_decorator(say_hello)`) likhna baar-baar thakau hai. Isliye Python **`@` syntax sugar** deta hai:

```python
def my_decorator(func):
    def wrapper():
        print("Function chalne se PEHLE...")
        func()
        print("Function chalne ke BAAD...")
    return wrapper

@my_decorator          # 👈 ye line bilkul yehi karti hai: say_hello = my_decorator(say_hello)
def say_hello():
    print("Hello Samarth!")

say_hello()    # ab seedha call karo, decorator automatically apply ho chuka hai
```

> **Sabse important cheez samajhne wali:** `@my_decorator` sirf ek shortcut hai. Ye internally exactly wahi karta hai jo `say_hello = my_decorator(say_hello)` karta — bas likhna aasaan bana deta hai.

```mermaid
flowchart TD
    A["@my_decorator\ndef say_hello(): ..."] -->|Python isko convert karta hai| B["def say_hello(): ...\nsay_hello = my_decorator(say_hello)"]
    B --> C["Ab 'say_hello' naam\nasal mein 'wrapper' function ko point karta hai"]
```

---

## 2.6 Arguments Wale Functions Decorate Karna

Problem: upar wala `wrapper()` koi argument accept nahi karta. Agar original function ko arguments chahiye, to decorator fail ho jaayega. Solution: `*args` aur `**kwargs` use karo, taaki wrapper **kisi bhi number of arguments** ko accept karke aage forward kar sake.

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):     # kitne bhi positional/keyword arguments accept karo
        print("Function chalne se PEHLE...")
        result = func(*args, **kwargs)   # unhi arguments ko original function ko forward karo
        print("Function chalne ke BAAD...")
        return result                      # original function ka result wapas return karo
    return wrapper

@my_decorator
def add(a, b):
    return a + b

print(add(5, 3))
# Output:
# Function chalne se PEHLE...
# Function chalne ke BAAD...
# 8
```

> Dhyan do: `result = func(*args, **kwargs)` aur `return result` bahut zaroori hai — inke bina decorated function ka return value **hamesha `None`** aayega, kyunki wrapper khud kuch return nahi kar raha tha.

---

## 2.7 `functools.wraps` — Metadata Save Karna

Ek subtle problem: jab tum function decorate karte ho, uska **naam aur docstring** "wrapper" ke naam se replace ho jaata hai:

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def add(a, b):
    """Do numbers add karta hai"""
    return a + b

print(add.__name__)   # 'wrapper' ❌ (expect kiya tha 'add')
print(add.__doc__)     # None ❌ (expect kiya tha 'Do numbers add karta hai')
```

**Fix:** `functools.wraps` decorator use karo — ye original function ka metadata (`__name__`, `__doc__`) wrapper pe copy kar deta hai:

```python
from functools import wraps

def my_decorator(func):
    @wraps(func)                  # 👈 ye line fix karti hai
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def add(a, b):
    """Do numbers add karta hai"""
    return a + b

print(add.__name__)   # 'add' ✅
print(add.__doc__)     # 'Do numbers add karta hai' ✅
```

> **Best practice:** Har decorator mein `@wraps(func)` use karne ki aadat daal lo — debugging aur documentation dono ke liye zaroori hai.

---

## 2.8 Decorators with Parameters (Decorator Factory)

Kabhi-kabhi decorator ko khud **apne parameters** chahiye hote hain (jaise "ye function 3 baar retry karo"). Iske liye ek **extra outer layer** chahiye hoti hai — isko **decorator factory** kehte hain.

```python
def repeat(times):                        # Level 1: ye "factory" hai, parameter leta hai
    def decorator(func):                   # Level 2: asli decorator
        @wraps(func)
        def wrapper(*args, **kwargs):      # Level 3: wrapper
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say_hi():
    print("Hi!")

say_hi()
# Output:
# Hi!
# Hi!
# Hi!
```

```mermaid
flowchart TD
    A["@repeat(times=3)"] --> B["repeat(3) call hota hai\nreturn karta hai 'decorator'"]
    B --> C["decorator(say_hi) call hota hai\n(jaisa normal decorator)"]
    C --> D["return karta hai 'wrapper'"]
    D --> E["say_hi ab wrapper ko point karta hai"]
```

> **3 layers yaad rakhne ka tareeka:** `repeat(3)` → parameters leta hai. Uska return (`decorator`) → asli function leta hai. Uska return (`wrapper`) → asli call handle karta hai.

---

## 2.9 Multiple Decorators Chain Karna

Ek function pe **multiple decorators** bhi laga sakte ho. Order **neeche se upar (bottom-to-top)** apply hota hai:

```python
def decorator_one(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Decorator ONE — pehle")
        result = func(*args, **kwargs)
        print("Decorator ONE — baad")
        return result
    return wrapper

def decorator_two(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Decorator TWO — pehle")
        result = func(*args, **kwargs)
        print("Decorator TWO — baad")
        return result
    return wrapper

@decorator_one
@decorator_two
def greet():
    print("Hello!")

greet()
```

**Output:**
```
Decorator ONE — pehle
Decorator TWO — pehle
Hello!
Decorator TWO — baad
Decorator ONE — baad
```

```mermaid
flowchart TD
    A["greet() call hua"] --> B["decorator_one ka wrapper shuru\n'ONE — pehle'"]
    B --> C["decorator_two ka wrapper shuru\n'TWO — pehle'"]
    C --> D["Original greet() chala\n'Hello!'"]
    D --> E["decorator_two ka wrapper khatam\n'TWO — baad'"]
    E --> F["decorator_one ka wrapper khatam\n'ONE — baad'"]
```

> **Yaad rakhne ka tareeka:** Ye bilkul **onion (pyaaz) ki layers** jaisa hai — jo decorator sabse **paas (bottom)** hai wo function ke sabse **paas** wrap karta hai (pehle chalta hai andar se), jo **upar** hai wo sabse **bahar** wrap karta hai (sabse pehle shuru, sabse aakhir khatam).

---

## 2.10 Class-based Decorators

Decorators sirf functions se nahi, **classes se bhi** ban sakte hain — bas class mein `__call__` method define karni hoti hai.

```python
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"'{self.func.__name__}' ab tak {self.count} baar call hua")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello!")

say_hello()   # 'say_hello' ab tak 1 baar call hua \n Hello!
say_hello()   # 'say_hello' ab tak 2 baar call hua \n Hello!
```

> Ye pattern useful hai jab decorator ko **state maintain** karni ho (jaise call count) — class attributes mein state rakhna function-based closures se zyada readable ho sakta hai.

---

## 2.11 Built-in Decorators

Python khud kuch bahut common decorators deta hai, jo classes ke saath use hote hain:

| Decorator | Kya karta hai | Example |
|---|---|---|
| `@staticmethod` | Function ko class ke andar rakho, par usko `self` na chahiye | Utility functions jo class ke data se independent hain |
| `@classmethod` | Function ko `self` ki jagah `cls` (class khud) milta hai | Alternate constructors banane ke liye |
| `@property` | Method ko **attribute ki tarah access** karne dega (bina `()` lagaye) | Computed values jo real attribute jaisi dikhein |

```python
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):                    # ab .area() nahi, seedha .area likhna hoga
        return 3.14159 * self.radius ** 2

    @staticmethod
    def is_valid_radius(r):             # self ki zaroorat nahi
        return r > 0

    @classmethod
    def unit_circle(cls):               # cls = Circle class khud
        return cls(radius=1)

c = Circle(5)
print(c.area)                     # 78.53975 — bina () ke, property ki tarah
print(Circle.is_valid_radius(5))  # True
c2 = Circle.unit_circle()         # radius=1 wala circle banaya
```

---

## 2.12 Real-World Use Cases

### a) Logging — function kab call hua, track karna

```python
import time
from functools import wraps

def log_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] '{func.__name__}' call ho raha hai...")
        result = func(*args, **kwargs)
        print(f"[LOG] '{func.__name__}' complete hua")
        return result
    return wrapper
```

### b) Timing — function kitna time leta hai, measure karna

```python
def measure_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"'{func.__name__}' ne {end - start:.4f} seconds liye")
        return result
    return wrapper
```

### c) Caching — same input ke liye dobara compute na karna (`functools.lru_cache`)

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def slow_fibonacci(n):
    if n <= 1:
        return n
    return slow_fibonacci(n - 1) + slow_fibonacci(n - 2)

print(slow_fibonacci(35))   # pehli baar slow, par results cache ho jaate hain
print(slow_fibonacci(35))   # dusri baar turant — cache se mila
```

### d) Authentication — check karna user allowed hai ya nahi (jaisa web APIs mein hota hai)

```python
def require_login(func):
    @wraps(func)
    def wrapper(user, *args, **kwargs):
        if not user.get("is_logged_in"):
            print("Access denied — login required!")
            return None
        return func(user, *args, **kwargs)
    return wrapper

@require_login
def view_dashboard(user):
    print(f"Welcome to dashboard, {user['name']}!")

view_dashboard({"name": "Samarth", "is_logged_in": True})    # Welcome...
view_dashboard({"name": "Guest", "is_logged_in": False})     # Access denied!
```

> Yehi exact pattern (`@require_login`, `@require_auth`) real backend frameworks (Flask, Django, FastAPI) mein API routes ko protect karne ke liye use hota hai.

---

## 2.13 Common Pitfalls

### a) `*args, **kwargs` bhoolna

```python
def bad_decorator(func):
    def wrapper():           # ❌ koi arguments accept nahi karta
        return func()
    return wrapper

@bad_decorator
def add(a, b):
    return a + b

add(2, 3)   # ❌ TypeError: wrapper() takes 0 positional arguments but 2 were given
```

### b) `return` bhoolna wrapper ke andar

```python
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)    # ❌ result store nahi kiya, return nahi kiya
    return wrapper

@bad_decorator
def add(a, b):
    return a + b

print(add(2, 3))   # None ❌ — expect kiya tha 5
```

### c) `@wraps` na lagana

Debugging mushkil ho jaati hai kyunki `func.__name__` hamesha `"wrapper"` dikhayega, asli function ka naam nahi.

```mermaid
flowchart TD
    A["Decorator likhte waqt\nkya check karein?"] --> B["1. wrapper mein *args, **kwargs hai?"]
    A --> C["2. func(*args, **kwargs) ka result\nreturn ho raha hai?"]
    A --> D["3. @wraps(func) laga hai?"]
```

---

## Quick Recap

```mermaid
%%{init: {'theme': 'dark', 'themeVariables': {'primaryColor': '#3b4252', 'primaryTextColor': '#eceff4', 'primaryBorderColor': '#88c0d0', 'lineColor': '#88c0d0', 'secondaryColor': '#434c5e', 'tertiaryColor': '#4c566a', 'mainBkg': '#2e3440', 'textColor': '#eceff4'}}}%%
mindmap
  root((Decorators +\nNamespaces))
    Namespaces
      "Naam to object mapping"
      Built-in / Global / Enclosing / Local
      LEGB Rule
      global keyword
      nonlocal keyword
      globals/locals
      Namespace lifecycle
    Decorators
      Functions are first-class
      Closures
      "@" syntax = shortcut
      "*args **kwargs"
      functools.wraps
      Decorator factory - parameters
      Multiple decorators - order
      Class-based decorators
      "staticmethod classmethod property"
      Logging Timing Caching Auth
```

**Bas itna samajh lo:**
- **Namespace** = naam → object ki mapping (alag-alag containers mein same naam alag cheez point kar sakta hai)
- **LEGB** = Python variable dhoondhne ka order: Local → Enclosing → Global → Built-in
- **Decorator** = ek function jo doosre function ko wrap karke uska behavior extend karta hai, bina uska code chhede
- **Closures** decorators ki foundation hain — inner function outer ke variables ko "yaad" rakhta hai
- **`@wraps(func)`** hamesha lagao, taaki original function ka naam/docstring preserve rahe

---

*Practice ke liye: ek `@retry(times=3)` decorator banao jo kisi function ko fail hone pe 3 baar retry kare (try-except ke saath), aur agar teeno baar fail ho to error raise kare.*


*  namespace is a space that holds names(identifiers).Programmatically speaking, namespaces are dictionary of identifiers(keys) and their objects(values)

There are 4 types of namespaces:
- Builtin Namespace
- Global Namespace
- Enclosing Namespace
- Local Namespace

In [ ]:
#scope ek aisa area hai jahan ek particular area is accessible 

#LEGB rule(local enclosing global builtin)

#interpreter ka searching order is local scope check->enclosing scope check -> global scope check ->and built in scope check 
# agar in chaaron scope main khi nhi mila to Name Error Exception dega

In [ ]:
# local and global
# global var because main program ke scope main haib
a = 2

def temp():
  # local var because main program ke andar jo function hai uska khud ka kuch alag part hai jo func ko belomg karta hai to to ye local hua 
  b = 3
  print(b)

temp()
print(a)

In [ ]:
#local ke pass to nhi hai ek variable par wo gloabl main hai 
a=2
def temp():
    print(a)

temp()
print(a)
# LEGB rule followed since yahan E hai hi nhi to direct global check hoga 
  

2
2


In [ ]:
a=2
def temp():
    a+=1 #local scope se tum global scope ko dekh skte ho changes nhi kar skte kyunki a+=1 is like a=a+1 and python dekhta hai  local var hai par isme a ki value to khi di hi nhi  global main hai to par usko access kar hi nhi paya 
    print(a)

temp()
print(a)

In [3]:
a=2
def temp():
    global a  #python says: "Don't create a new, local version of a inside this function. Instead, look outside the function and use the global variable a that already exists." isse gloabl var a ki value bhi change ho jayegi .bUT this is not good programming practise 
    a+=1
    print(a)

temp()
print(a)

3
3


In [ ]:
def temp():
    global a  #Python bolega ki anytime ab main a cretae karunga to usko global bana dunga lets you create a global variable right from inside the function
    a=1
    print(a)

temp()
print(a)
#ab kya hoga to hoga yeh yeh ki global a se ye hua  ki jo fucntion ke andar a bana na woh global bana 

In [ ]:
# local and global -> function parameter is local kyunki global se copy aati hai yni global se jo parameter bhejte hai woh khud nhi aata apni copy ko bhejta hai to us copy ko store karne ke liye var to lagega hi na 

def temp(z):
  # local var
  print(z)

a = 5
temp(5)
print(a)
print(z) 

In [ ]:
#Built in scope 
print() #Is perfect example of built in scope. Ab python kya karta hai jab hum code likhte hai na to usse pehle hi kuch cheezein like available karwa deta hai ki bindaas use karo bhai koi dikkat nhi jaise input(),print(),int,eval oto h=yhi cheezein built in scope kehlati hai 

#Built in main kya kya hai # built-in scope
import builtins
print(dir(builtins))#ways to see 


['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarnin

In [ ]:
# renaming built-ins
L = [1,2,3]
print(max(L))
def max():
  print('hello')

print(max(L))

# ab dekho max() ek list ka function bhi hota hai and humne bhi max bana diya to kounsa chalega ? kya error aayega ? to haaan error aayega par kyu?? to isiliye kyunki LEGB rule ke hissab se pehle local dekho phir enclosed dekho fir global dekho and fir built dekho yahan jab global main hii max mila kyu built in main kyu ajey  also built in ke max function koi arguent nhi le raha lekin hum zabardasti pass kr rhe isliye error 

In [ ]:
#Enclosing scope matlab helps you to see inside nested functions 
#Enclosing scope :function ke andar functions 

# Enclosing scope
def outer():
  def inner():
    print("inner funncitons")
  inner()
  print('outer function')


outer()
print('main program') #are seedhe matlab ek example se samjho 

3
outer function
main program


In [ ]:

def outer():
  a=3
  def inner():
    a=4
    print(a)
  inner()
  print('outer function')

a=5
outer()
print('main program')
#kOunsa a print hoga to outer call hua then inner call hua then inner ka print (a ) aaya to it searches pehle local scope matlab inner func main dhundhega a hai ki nhi hai to prinnt kar diya agar nhi hota a to fir inner ko enclose koun kar raha hai  outer to outer ka local scope inner ke liye enclosing scope hoga then is scope main a=3 hai 3 print hoagyaa aga ryhan nhi hota to global dekhta outer ke bahar a=5 hai 5 print hota 

In [ ]:
#local se global main hum change kar skte hai usi prakar inner se enclosing main bhi change kar skte hai 

# nonlocal keyword
def outer():
  a = 1
  def inner():
    nonlocal a
    a += 1
    print('inner',a)
  inner()
  print('outer',a)


outer()
print('main program')

#ab nonlocal a bolta hai apne enclosing scope main a hai to usko apna hi man 

### Decorators

A decorator in python is a function that receives another function as input and adds some functionality(decoration) to and it and returns it.

This can happen only because python functions are 1st class citizens.

There are 2 types of decorators available in python
- `Built in decorators` like `@staticmethod`, `@classmethod`, `@abstractmethod` and `@property` etc
- `User defined decorators` that we programmers can create according to our needs

In [ ]:
# Python are 1st class function

def modify(func,num):
  return func(num)

def square(num):
  return num**2

num=modify(square,2)
print(num)
#jaise yahan dekho hum modify ko square function (actually sqaure ka reference ) pass kar rhe hai decorator main bhi yhi hoga 

4


In [ ]:
#Simple example :- ek decorator banao jo output ko legit decorate akarde by maing lines on top and bottom of it 

def my_decorator(func): #custom decorator hai ye 
  def wrapper(): #custom decorator/decorator ke adnar ek wrapper function hota hi hai jiska nam legit wrapper hi hota hai 
    print('***********************')
    func()
    print('***********************')
  return wrapper #wrapper func ka reference return kar diya 

def hello():
  print('hello')

def display():
  print('hello nitish')

a = my_decorator(hello)  #decorator ko function diya aur return main wrapper mil gaya 
a() #wrapper call hogaya 
#Ab koi bhi function ko decorate kar skta hai my_decorator ka use karke
b = my_decorator(display)
b()

#USes concepts of closures abhi nhi padhoge koi dikkat nhi frontend ke time js padhoge to wahan padh hi loge 

***********************
hello
***********************
***********************
hello nitish
***********************


In [ ]:
#jaise hum likh rhe the wo bohot hi inconveinient hai bhai isse ache custom decorator banao and phir jo function as an input jayega uske upar @(custom decorator ka naam) ye synatax likh do kaam done 

#Example:-

def my_decorator(func):
  def wrapper():
    print('***********************')
    func()
    print('***********************')
  return wrapper

@my_decorator
def hello():
  print('hello')

hello()

In [ ]:
#Logical Decorator Example:-
#YE decorator kisi bhi function ke execute hone ke time ko batayega 
import time

def timer(func):
  def wrapper():
    start = time.time()
    func()
    print('time taken by',func.__name__,time.time()-start,'secs') 
    #func.__name__ se function ka actual naam return hojayega 

  return wrapper

@timer
def hello():
  print('hello wolrd')
  time.sleep(2)

hello()


In [ ]:

import time

def timer(func):
  def wrapper(*args):
    start = time.time()
    func()
    print('time taken by',func.__name__,time.time()-start,'secs')
  return wrapper

# @timer
# def hello():
#   print('hello wolrd')
#   time.sleep(2)

@timer
def square(num):
  time.sleep(1)
  print(num**2)



# hello()
square(2)

# ab error aajayega kyu kyunki pehle timer call hoga and uske pass sqare jayega as argument now wrapper return aayega then wrappper call hoga and finally jab func call hoga jiske pass sqaure ka reference hai to it will require num as argument par argument to tune diya hi nhi na matlab func ke pas square ka reference par argument nhi wwhile calling 
#code is not generic kyunki ye sirf unhi ko call karega jinke paa=ss ek bhi nhi hai argument to generalize karna hoga

hello wolrd
time taken by hello 2.0013558864593506 secs
4
time taken by square 1.0009427070617676 secs


In [13]:
#More generalized format
import time

def timer(func):
  def wrapper(*args):
    start = time.time()
    func(*args)
    print('time taken by',func.__name__,time.time()-start,'secs')
  return wrapper

@timer
def hello():
  print('hello wolrd')
  time.sleep(2)

@timer
def square(num):
  time.sleep(1)
  print(num**2)

@timer
def power(a,b):
  print(a**b)

hello()
square(2)
power(2,3)


hello wolrd
time taken by hello 2.001237154006958 secs
4
time taken by square 1.0009324550628662 secs
8
time taken by power 3.1948089599609375e-05 secs


In [ ]:
#Second useful decorator :will check whether function ke andar mila datatype is correct or not 

#IS decoraator  ko a func ke alwa kuch aur bhi input lagega 

def sanity_check(data):
    def parent_wrapper(func):
        def wrapper(*args):
            if type(*args)==data :
                return func(*args)
            else:
                raise TypeError("Galat datatype diya re ")  
        return wrapper    
    return parent_wrapper    
             
@sanity_check(int)
def square(num):
    return num**2

print(square(9))

81
